# OmniTrainer — Demo Notebook

End-to-end walkthrough of the multimodal content moderation pipeline.
Runs against **TestModel** (no Gemini API key required for sections 3–7).


## 1  Setup

Sync dependencies and install the package. These commands use  to run
from the project root (one level above this notebook).


In [1]:
import subprocess, sys
from pathlib import Path

# Project root is one directory above this notebook
ROOT = Path("__file__").resolve().parent.parent if "__file__" in dir() else Path.cwd().parent
ROOT = ROOT if (ROOT / "pyproject.toml").exists() else Path.cwd().parent

# Install / sync dependencies
!cd "{ROOT}" && uv sync --dev -q
!cd "{ROOT}" && uv pip install -e . -q

## 2  Run the Test Suite

All tests use  — no real API calls, no  needed.


In [2]:
!cd "{ROOT}" && uv run pytest tests/ -q

............................................................             [100%]
60 passed, 1 deselected in 18.90s


## 3  Imports and Configuration


In [3]:
import os, sys
from pathlib import Path

# Ensure project root is on sys.path so multimodal_moderation can be imported
# even if the package was not installed via pip install -e .
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Set dummy credentials (replace with your real key to call Gemini).
os.environ.setdefault("GEMINI_API_KEY", "your-gemini-api-key")
os.environ.setdefault("USER_API_KEY", "demo-key")
os.environ.setdefault("DEFAULT_GOOGLE_MODEL", "gemini-2.5-flash-lite")

from pydantic_ai import models
from pydantic_ai.models.test import TestModel

from multimodal_moderation.agents.text_agent import moderate_text, text_moderation_agent
from multimodal_moderation.agents.image_agent import moderate_image, image_moderation_agent
from multimodal_moderation.agents.audio_agent import moderate_audio, audio_moderation_agent
from multimodal_moderation.agents.video_agent import moderate_video, video_moderation_agent
from multimodal_moderation.env import get_default_model_choice

models.ALLOW_MODEL_REQUESTS = False  # remove this line to call the real Gemini API
model = get_default_model_choice()

# test_data lives at <root>/tests/test_data/
TEST_DATA = ROOT / "tests" / "test_data"
print("Imports OK")
print(f"TEST_DATA: {TEST_DATA}")

Imports OK
TEST_DATA: /Users/suleimanadebowaleojo/Claude/Projects/OmniTrainer Multimodal Customer Trainer/tests/test_data


## 4  Text Moderation


In [4]:
professional_msg = "Thank you for reaching out! I'd be happy to help."

with text_moderation_agent.override(model=TestModel()):
    result = await moderate_text(model, professional_msg)

print("Text   :", professional_msg)
print("Flagged:", result.is_flagged())
print("Result :", result.model_dump())

Text   : Thank you for reaching out! I'd be happy to help.
Flagged: False
Result : {'rationale': 'a', 'contains_pii': False, 'is_unfriendly': False, 'is_unprofessional': False}


## 5  Image Moderation


In [5]:
image_bytes = (TEST_DATA / "simple_image.jpg").read_bytes()

with image_moderation_agent.override(model=TestModel()):
    img_result = await moderate_image(model, image_bytes, "image/jpeg")

print("Flagged:", img_result.is_flagged())
print("Result :", img_result.model_dump())

Flagged: False
Result : {'rationale': 'a', 'contains_pii': False, 'is_unfriendly': False, 'is_unprofessional': False, 'is_disturbing': False, 'is_low_quality': False}


## 6  Audio Moderation


In [6]:
audio_bytes = (TEST_DATA / "simple_audio.mp3").read_bytes()

with audio_moderation_agent.override(model=TestModel()):
    aud_result = await moderate_audio(model, audio_bytes, "audio/mpeg")

print("Transcription:", aud_result.transcription)
print("Flagged      :", aud_result.is_flagged())
print("Result       :", aud_result.model_dump())

Transcription: a
Flagged      : False
Result       : {'rationale': 'a', 'contains_pii': False, 'is_unfriendly': False, 'is_unprofessional': False, 'transcription': 'a'}


## 7  Moderation Result Schemas

All four agents return Pydantic models sharing a common  base.


In [7]:
from multimodal_moderation.types.moderation_result import (
    TextModerationResult, ImageModerationResult,
    AudioModerationResult, VideoModerationResult,
)

for cls in [TextModerationResult, ImageModerationResult,
            AudioModerationResult, VideoModerationResult]:
    schema = cls.model_json_schema()
    print(f"{cls.__name__} ---")
    print("Fields:", list(schema["properties"].keys()))

TextModerationResult ---
Fields: ['rationale', 'contains_pii', 'is_unfriendly', 'is_unprofessional']
ImageModerationResult ---
Fields: ['rationale', 'contains_pii', 'is_unfriendly', 'is_unprofessional', 'is_disturbing', 'is_low_quality']
AudioModerationResult ---
Fields: ['rationale', 'contains_pii', 'is_unfriendly', 'is_unprofessional', 'transcription']
VideoModerationResult ---
Fields: ['rationale', 'contains_pii', 'is_unfriendly', 'is_unprofessional', 'is_disturbing', 'is_low_quality']


## 8  Next Steps

- Set a real  in  and remove the  line above to call Gemini
- From the project root:  to start all three services
- Chat UI   → http://localhost:7860
- Phoenix   → http://localhost:6006
- API docs  → http://localhost:8000/docs
- Run evals: 
